In [ ]:
##请先确定环境里用安装changeo和biopython的包
pip install changeo --user
pip install biopython

In [ ]:
##add igblastn to your PATH, do this on the terminal
!export PATH="/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo/igblast/ncbi-igblast-1.21.0/bin:$PATH"

In [13]:
##add path to the jupyter notebook
import os
os.environ["PATH"] = "/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo/igblast/ncbi-igblast-1.21.0/bin:" + os.environ["PATH"]


In [14]:
!which igblastn

/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo/igblast/ncbi-igblast-1.21.0/bin/igblastn


In [28]:
# ##这一步在跑igblast,会很慢
!python ./AssignGenes.py igblast \
    -s ./input/141415_7.fasta -b igblast -o ./temp/Tar_igblast.fmt7

#这一步速度还行,在归纳igblast结果
!python ./MakeDb.py igblast  \
    -s ./input/141415_7.fasta -r igblast/human/vdj \
    -i ./temp/Tar_igblast.fmt7 -o ./temp/Tar_igblast_db-pass.tsv

#这一步也超级慢，进行一个基于CDR3和gene_usage的聚类
!python ./DefineClones.py   \
    -d ./temp/Tar_igblast_db-pass.tsv --mode gene --act set\
    --model hh_s5f --dist 0.2  \
    --sf JUNCTION --norm len \
    -o ./temp/Tar_igblast_db-pass_clone-pass.tsv

#这一步给每个类计算一个germline,速度很快
!python ./CreateGermlines.py   \
    -d ./temp/Tar_igblast_db-pass_clone-pass.tsv  -r igblast/human/vdj    \
    -g dmask --cloned  \
    -o ./temp/Tar_igblast_db-pass_clone-pass_germ-pass.tsv

!python ./CreateGermlines.py   \
    -d ./temp/Tar_igblast_db-pass_clone-pass.tsv  -r igblast/human/vdj    \
    -g dmask --cloned  \
    -o ./output/141415_7.tsv

/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo/./AssignGenes.py:14: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
WARNING> Output file ./temp/Tar_igblast.fmt7 already exists and will be overwritten.
   START> AssignGenes
 COMMAND> igblast
 VERSION> 1.22.0
    FILE> 141415_7.fasta
ORGANISM> human
    LOCI> ig
   NPROC> 32

PROGRESS> 11:43:17 |Done                     | 0.8 min

  PASS> 24244
OUTPUT> Tar_igblast.fmt7
   END> AssignGenes

WARNING> Output file ./temp/Tar_igblast_db-pass.tsv already exists and will be overwritten.
         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> Tar_igblast.fmt7
      SEQ_FILE> 141415_7.fasta
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> False
INFER_JUNCTION> False

PROGRESS> 11:43:17 |Done                | 0.0 min

PROGRESS> 11:43:25 |####################| 100% (24,244) 0.

In [54]:
import pandas as pd
from Bio.Seq import Seq
import glob

In [46]:
# Load the CSV

df=pd.read_csv("./output/120648_0.tsv",sep="\t")
df.to_csv("./output/120648_0.csv",sep=",")

In [47]:
df.iloc[0,:]

sequence_id                                        ACGCAGCTCAGGCAAG-1_contig_1
sequence                     GGGAGCATCACATAACAACCACATTCCTCCTCTAAAGAAGCCCCTG...
rev_comp                                                                     F
productive                                                                   T
v_call                                                IGHV1-69*01,IGHV1-69D*01
d_call                                                 IGHD1-26*01,IGHD6-25*01
j_call                                                                IGHJ6*02
sequence_alignment           CAGGTGCAGCTGGTGCAGTCTGGGGCT...GAGGTGAAGAAGCCTG...
germline_alignment           CAGGTGCAGCTGGTGCAGTCTGGGGCT...GAGGTGAAGAAGCCTG...
junction                                  TGTGCGAGAGAGGAGCGGTACGGTATGGACGTCTGG
junction_aa                                                       CAREERYGMDVW
v_cigar                                                               121S296=
d_cigar                                             

In [48]:
# Extract the barcode (everything before first '_contig_')
df['barcode'] = df['sequence_id'].str.replace(r'_contig_.*$', '', regex=True)
print(df[['sequence_id', 'barcode']].head())

                   sequence_id             barcode
0  ACGCAGCTCAGGCAAG-1_contig_1  ACGCAGCTCAGGCAAG-1
1  CTGTGCTGTGACGCCT-1_contig_4  CTGTGCTGTGACGCCT-1
2  CGATGTAGTTACTGAC-1_contig_2  CGATGTAGTTACTGAC-1
3  GCTCTGTTCTTATCTG-1_contig_3  GCTCTGTTCTTATCTG-1
4  ACTTGTTGTAGCAAAT-1_contig_1  ACTTGTTGTAGCAAAT-1


In [49]:
# Remove gaps/dots and translate
def translate_alignment(nt):
    if pd.isnull(nt):
        return ""
    nt_clean = nt.replace(".", "").replace("-", "")  # Remove IMGT dots and dashes
    nt_clean = nt_clean.upper().replace(' ', '').replace('\n','')
    # Optionally, ensure length is a multiple of 3 (trim excess)
    nt_clean = nt_clean[:len(nt_clean) - len(nt_clean) % 3]
    try:
        aa = str(Seq(nt_clean).translate())
        return aa
    except Exception as e:
        return ""

# Create amino acid column
df['sequence_aa'] = df['sequence_alignment'].apply(translate_alignment)

print("✅ Example nucleotide and AA:")
print(df[['sequence_alignment','sequence_aa']].head())

✅ Example nucleotide and AA:
                                  sequence_alignment  \
0  CAGGTGCAGCTGGTGCAGTCTGGGGCT...GAGGTGAAGAAGCCTG...   
1  GAGGTGCAGCTGGTGGAGTCTGGAGGA...GGCTTGATCCAGCCTG...   
2  GTCATCTGGATGACCCAGTCTCCATCCTTACTCTCTGCATCTACAG...   
3  GAGGTGCAGCTGGTGCAGTCTGGAGCA...GAGGTGAAAAAGCCCG...   
4  GAGGTGCAGCTGGTGCAGTCTGGAGCA...GAGGTGAAAAAGCCCG...   

                                         sequence_aa  
0  QVQLVQSGAEVKKPGSSVKVSCKASGGTFSSYAISWVRQAPGQGLE...  
1  EVQLVESGGGLIQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...  
2  VIWMTQSPSLLSASTGDRVTISCRMSQGISSYLAWYQQKPGKAPEL...  
3  EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...  
4  EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLE...  


In [50]:
# Subset heavy and light chains
df_H = df[df['locus'] == 'IGH'].copy()
df_L = df[df['locus'].isin(['IGK','IGL'])].copy()

In [51]:
# Rename columns for merging
df_H = df_H.rename(columns={'sequence_alignment':'H_nt', 'sequence_aa':'H_aa'})
df_L = df_L.rename(columns={'sequence_alignment':'L_nt', 'sequence_aa':'L_aa'})
# Keep only relevant columns and barcode
df_H = df_H[['barcode', 'H_nt', 'H_aa', 'c_call','clone_id']]
df_L = df_L[['barcode', 'L_nt', 'L_aa', 'c_call','clone_id']]

In [52]:
# Pair heavy and light by barcode (inner join)
paired = pd.merge(df_H, df_L, on='barcode', how='inner')


In [53]:
# Save paired result to TSV
paired.to_csv("paired_HL120648_0.tsv", sep="\t", index=False)


In [55]:
#concatenate the files
# Path to your folder containing tsv files
folder = "./"  # or specify your absolute path

# Get all .tsv files in the folder
tsv_files = glob.glob(os.path.join(folder, "*.tsv"))
all_dfs = []

In [56]:
for file in tsv_files:
    sample_name = os.path.basename(file).replace('.tsv', '')
    df = pd.read_csv(file, sep="\t")
    df['sample'] = sample_name
    all_dfs.append(df)

# Concatenate all
if all_dfs:
    big_df = pd.concat(all_dfs, ignore_index=True)
    # Save to CSV
    big_df.to_csv("all_samples_combined.csv", index=False)
    print(f"✅ Combined {len(tsv_files)} files and saved as all_samples_combined.csv")
    print(big_df.head())
else:
    print("No TSV files found in the folder!")

✅ Combined 12 files and saved as all_samples_combined.csv
              barcode                                               H_nt  \
0  CCCAATCGTGGCGAAT-1  GAGGTGCAGCTGGTGGAGTCTGGGGGA...GGCTTGGTAAAGCCTG...   
1  ACCGTAATCAACACGT-1  GAGGTGCAGCTGTTGGAGTCTGGGGGA...GGCTTGGTACAGCCTG...   
2  CACAGGCAGGCAAAGA-1  GAGGTGCAGCTGTTGGAGTCTGGGGGA...GGCTTGGTACAGCCTG...   
3  CACAGGCAGGCAAAGA-1  GAGGTGCAGCTGTTGGAGTCTGGGGGA...GGCTTGGTACAGCCTG...   
4  CACAGGCAGGCAAAGA-1  GAGGTGCAGCTGTTGGAGTCTGGGGGA...GGCTTGGTACAGCCTG...   

                                                H_aa         c_call_x  \
0  EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWMSWVRQAPGKGLE...  IGHM*01,IGHM*03   
1  EVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLE...  IGHM*01,IGHM*03   
2  EVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLE...  IGHD*01,IGHD*02   
3  EVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLE...  IGHD*01,IGHD*02   
4  EVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLE...  IGHD*01,IGHD*02   

   clone_id_x  sample_x                       